# Featurizer Tutorial: Temporal (As-Of) Joins

This notebook demonstrates temporal joins in Featurizer using a healthcare scenario:
- **Patients** (target entity) with admission dates
- **Care Plans** (related entity) with plan dates

The key concept is the **as-of join**: for each patient row, we want features from the most recent care plan *as of* that patient's admission date.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

if not Path("data.db").exists():
    exec(open("create_data.py").read())

## 2. Understanding Temporal Relationships

The configuration defines a temporal relationship with `mode: as_of`:

In [ ]:
with open("config.yaml") as f:
    print(f.read())

### Key Temporal Configuration:

```yaml
temporal:
  mode: as_of        # Enable as-of join semantics
  grace: P7D         # Allow matching up to 7 days in the future
  child_time: plan_date  # Override child entity's temporal column
```

This means: for each patient admission, find the most recent care plan where `plan_date <= admission_date + 7 days`.

## 3. Load Configuration

In [ ]:
from featurizer import Featurizer

featurizer = Featurizer("config.yaml")

print(f"Target: {featurizer.target.alias}")
print(f"Intervals: {featurizer.intervals}")

In [ ]:
# Examine the temporal relationship
for rel in featurizer.relationships:
    print(f"Relationship: {rel.parent.alias} -> {rel.child.alias}")
    print(f"  Temporal mode: {rel.temporal_mode}")
    print(f"  Grace period: {rel.temporal_grace}")
    print(f"  Child timestamp: {rel.temporal_child_field}")

## 4. Examining the Generated SQL

The key difference from regular joins is the `LEFT JOIN LATERAL` clause.

In [ ]:
sql = featurizer.query

# Find the LATERAL join
if 'lateral' in sql.lower():
    print("Temporal join generates LEFT JOIN LATERAL:")
    print("=" * 60)
    
    # Extract the lateral join portion
    lines = sql.split('\n')
    in_lateral = False
    for line in lines:
        if 'lateral' in line.lower():
            in_lateral = True
        if in_lateral:
            print(line)
            if ') as' in line.lower() and 'on true' in line.lower():
                break

### Understanding the LATERAL Join

The generated SQL uses a pattern like:

```sql
LEFT JOIN LATERAL (
    SELECT care_plans_transform.cost, ...
    FROM care_plans_transform
    WHERE care_plans_transform.patient_id = patients.patient_id
      AND care_plans_transform.plan_date <= patients.admission_date
      AND patients.admission_date - care_plans_transform.plan_date <= interval 'P7D'
    ORDER BY care_plans_transform.plan_date DESC
    LIMIT 1
) AS care_plans_asof_for_patients ON true
```

This ensures:
1. Only matching patient_id
2. Care plan date is before or at admission date
3. Within the grace period
4. Most recent match (ORDER BY ... DESC LIMIT 1)

## 5. Generated Features

In [ ]:
target_features = featurizer.features[featurizer.target.alias]

print(f"Total features: {len(target_features)}")
print("\nFeatures from care_plans (via temporal join):")
care_plan_features = [f for f in target_features if 'care_plans' in f.name.lower()]
for f in sorted(care_plan_features, key=lambda x: x.name)[:15]:
    print(f"  {f.name}")

## 6. When to Use Temporal Joins

Use temporal (as-of) joins when:

1. **Point-in-time correctness matters**: You need features as they would have been at a specific moment
2. **Preventing data leakage**: In ML, you can't use future data to predict past events
3. **Time-varying relationships**: The "current" related record changes over time

### Examples:
- Customer's current address at time of order
- Patient's latest diagnosis at time of treatment
- Product's price at time of purchase
- Employee's department at time of performance review

In [ ]:
print("Summary")
print("=" * 40)
print(f"Target: {featurizer.target.alias}")
print(f"Temporal relationship: patients <- care_plans (as-of)")
print(f"Grace period: P7D")
print(f"Total features: {len(target_features)}")